# YOLOv2 448 + crop, flip, brightness, cosine decay

**RECONSTRUCTED — this is not the original notebook.**

The original was never saved: the augmentation and schedule changes were
applied as patch cells inside a live Kaggle session, and the notebook was
downloaded before those cells were added. The run itself did happen, and
its weights are in `Data_new/runs/yolov2_rgb_448_aug2/`.

This file reproduces what that session did, assembled from the session log.
Cells 0-4 are copied verbatim from `yolov2_rgb_448_kaggle.ipynb`, which is a
real saved notebook. The patch cell and the training command are the
reconstructed part.

**Result: val AP 0.669, test AP 0.677.** Best training curves of any run, and worse than plain 448 on both splits.

**Platform:** Kaggle, T4 GPU, internet on.
**Dataset:** `tensors_rgb_448_packed` uploaded as a Kaggle dataset, together
with `config.py`, `dataset.py`, `targets.py`, `train.py`, `evaluate.py`.


In [ ]:
!nvidia-smi
!ls -R /kaggle/input | head -20

In [ ]:
BASE = '/kaggle/input/datasets/bornamuzina/dataset448'
DATA = BASE + '/tensors_rgb_448_packed_k/tensors_rgb_448_packed'
!cp {BASE}/*.py /kaggle/working/
%cd /kaggle/working
!ls *.py

In [ ]:
!apt-get install -qq python3.11 python3.11-venv python3.11-dev > /dev/null 2>&1
!python3.11 -m venv /kaggle/working/akv
!/kaggle/working/akv/bin/pip install -q --upgrade pip
!/kaggle/working/akv/bin/pip install -q akida-models==1.14.2

In [ ]:
import re
src = open('config.py').read()
src = re.sub(r"^ROOT = Path\(.*?\)$", "ROOT = Path('/kaggle/working')", src, flags=re.M)
src = re.sub(r"^TENSOR_DIR = .*$", f"TENSOR_DIR = Path('{DATA}')", src, flags=re.M)
src = re.sub(r"^SPLITS_FILE = .*$", f"SPLITS_FILE = Path('{DATA}/splits.json')", src, flags=re.M)
src = re.sub(r"^RUNS_DIR = .*$", "RUNS_DIR = Path('/kaggle/working/runs')", src, flags=re.M)
open('config.py','w').write(src)

src = open('dataset.py').read()
src = src.replace(
    'if not npy.exists() or not meta_path.exists():',
    'npz = tensor_dir / f"{clip}_tensors.npz"\n    if not meta_path.exists() or (not npy.exists() and not npz.exists()):'
)
src = src.replace(
    'tensors = np.load(npy, mmap_mode="r")',
    'tensors = np.load(npy, mmap_mode="r") if npy.exists() else np.load(npz)["a"]'
)
open('dataset.py','w').write(src)

!grep -n "ROOT\|TENSOR_DIR\|SPLITS_FILE\|RUNS_DIR\|INPUT_SIZE\|^GRID" config.py

In [ ]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python -u dataset.py

## The patch

Two changes on top of the flip/brightness version: random crop with rescale,
and cosine learning-rate decay.

The crop was the point. A flipped frame keeps the drone at the same position
and size, so a mirrored clip is nearly as memorisable as the original.
Cropping moves it and changes its apparent scale, which is the variation 90
fixed camera setups do not otherwise provide.

Boxes shift by the crop offset then scale by `size/window`. A box whose centre
leaves the window is dropped; if that empties the frame, the crop is undone
rather than training on an image labelled as containing a drone it no longer
contains.

Cosine decay was added because validation loss had been swinging between 1.68
and 2.20 on adjacent epochs — a flat learning rate stepping past good
solutions rather than settling into one.

In [ ]:
new_aug = """MIN_BOX = 6.0


def augment_batch(images, boxes_list, size=None, rng=np.random):
    \"\"\"
    Horizontal flip, brightness, and random crop/scale. Training only.

    The crop is what actually fights memorisation. Flips leave the drone
    at the same position and size, so a mirrored clip is nearly as
    memorisable as the original. Cropping moves it and changes its
    apparent scale -- the variation that 90 fixed camera setups do not
    otherwise provide.
    \"\"\"
    if size is None:
        size = config.INPUT_SIZE

    out_images = []
    out_boxes = []

    for img, boxes in zip(images, boxes_list):
        img = np.asarray(img, dtype=np.float32)
        original = img
        boxes = list(boxes)

        # ---- random crop and rescale -------------------------------
        # Zoom in only. Zooming out would need padding, and the frame is
        # already letterboxed, so it would stack padding on padding.
        if rng.random() < 0.7:
            frac = rng.uniform(0.6, 1.0)
            win = int(size * frac)
            x0 = rng.randint(0, size - win + 1)
            y0 = rng.randint(0, size - win + 1)

            crop = img[y0:y0 + win, x0:x0 + win]
            cropped = cv2.resize(crop, (size, size),
                                 interpolation=cv2.INTER_LINEAR)
            s = size / win

            moved = []
            for (x, y, w, h) in boxes:
                nx = (x - x0) * s
                ny = (y - y0) * s
                nw = w * s
                nh = h * s

                # A box whose centre left the window is gone. Keeping it
                # would teach the model to predict a drone that is not
                # in the image.
                cx, cy = nx + nw / 2.0, ny + nh / 2.0
                if not (0.0 <= cx < size and 0.0 <= cy < size):
                    continue

                x1 = max(nx, 0.0)
                y1 = max(ny, 0.0)
                x2 = min(nx + nw, float(size))
                y2 = min(ny + nh, float(size))

                # A sliver of wingtip still labelled "drone" teaches the
                # wrong thing, so drop anything smaller than MIN_BOX.
                if x2 - x1 >= MIN_BOX and y2 - y1 >= MIN_BOX:
                    moved.append((x1, y1, x2 - x1, y2 - y1))

            if boxes and not moved:
                img = original            # crop lost everything, undo it
            else:
                img = cropped
                boxes = moved

        # ---- horizontal flip ---------------------------------------
        if rng.random() < 0.5:
            img = img[:, ::-1]
            boxes = [(size - x - w, y, w, h) for (x, y, w, h) in boxes]

        # ---- brightness and contrast -------------------------------
        if rng.random() < 0.5:
            gain = rng.uniform(0.7, 1.3)
            bias = rng.uniform(-25.0, 25.0)
            img = np.clip(img * gain + bias, 0.0, 255.0)

        out_images.append(np.ascontiguousarray(img))
        out_boxes.append(boxes)

    return np.stack(out_images), out_boxes
"""

src = open('train.py').read()

if 'import cv2' not in src:
    src = src.replace('import numpy as np', 'import cv2\nimport numpy as np', 1)

start = src.index('def augment_batch(')
end = src.index('\n# ============================================================\n# LOSS\n')
src = src[:start] + new_aug + src[end:]

src = src.replace(
    "    optimizer = tf.keras.optimizers.Adam(learning_rate=args.lr)",
    "    # Flat lr made validation loss swing between 1.68 and 2.20 on\n"
    "    # adjacent epochs -- the model stepping past good solutions.\n"
    "    # Cosine decay to 1% of the initial rate lets it settle.\n"
    "    steps_per_epoch = max(1, len(train_ds) // args.batch_size)\n"
    "    schedule = tf.keras.optimizers.schedules.CosineDecay(\n"
    "        initial_learning_rate=args.lr,\n"
    "        decay_steps=steps_per_epoch * args.epochs,\n"
    "        alpha=0.01,\n"
    "    )\n"
    "    optimizer = tf.keras.optimizers.Adam(learning_rate=schedule)"
)

open('train.py','w').write(src)

!grep -n "import cv2\|def augment_batch\|CosineDecay\|if training:" train.py

## Verify before training

The crop can silently produce out-of-bounds or degenerate boxes, and a wrong
target is invisible during training — it just makes the run worse. 2,000
random trials, every box checked against the frame bounds.

Must print `bad: 0`. It did.

In [ ]:
test = """
import numpy as np, config, train as t
S = config.INPUT_SIZE
rng = np.random.RandomState(0)
bad = 0
for trial in range(2000):
    img = np.zeros((1, S, S, 3), np.float32)
    w = float(rng.randint(13, 40)); h = w / 1.3
    x = float(rng.randint(0, int(S - w))); y = float(rng.randint(45, int(S - h - 45)))
    img[0, int(y):int(y+h), int(x):int(x+w)] = 200.0
    a, b = t.augment_batch(img, [[(x, y, w, h)]], rng=rng)
    if a.min() < 0 or a.max() > 255: print("PIXEL RANGE"); bad += 1
    for (bx, by, bw, bh) in b[0]:
        if bx < -1e-6 or by < -1e-6 or bx+bw > S+1e-6 or by+bh > S+1e-6:
            print(f"OUT OF BOUNDS {bx:.1f} {by:.1f} {bw:.1f} {bh:.1f}"); bad += 1
print("bad:", bad)
"""
open('test_aug.py','w').write(test)

In [ ]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python test_aug.py

In [ ]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python -u train.py --epochs 25 --lr 1e-3 --batch_size 32 --name full_rgb_448_aug2

In [ ]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python -u evaluate.py --run full_rgb_448_aug2 --split validation

In [ ]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python -u evaluate.py --run full_rgb_448_aug2 --split test
!cd /kaggle/working && zip -qr full_rgb_448_aug2.zip runs/full_rgb_448_aug2
!ls -lh /kaggle/working/full_rgb_448_aug2.zip

## Result

| | val | test |
|---|---|---|
| AP@0.5 | 0.669 | 0.677 |
| precision | 0.764 | 0.749 |
| recall | 0.802 | 0.766 |

**Every training signal said this was the best run.** Val loss 1.169 at epoch 7
against the plain run's 2.074, val recall peaking at 0.904, and a train/val
recall gap of 0.013 where the plain run reached 0.29 by epoch 5. Overfitting
only returned after epoch 15.

**The AP said otherwise.** Plain 448 scored 0.711 on test; this scored 0.677.

The likely cause is a flaw in the crop: the window only zooms *in*, 60 to 100%
of the frame and never wider, because zooming out would need padding on an
already letterboxed frame. So every crop makes the drone larger, and the model
trained on a size distribution shifted away from the real one. Validation
recall by size shows it directly — the 8-12 px bin went 0.600 (plain) → 0.307
(flip) → 0.080 (crop), while 16+ px improved 0.826 → 0.859 → 0.878.

The model learned the augmented distribution rather than the real one. Fixing
this means letting the crop zoom out as well as in.

**Note:** the weights for this run were lost when the Kaggle session ended.
Only the numbers survive.